# **<U>B9DA106 DATA VISUALISATION**

---



# CA-2 Assessment

## 1).   **Introduction**


In this analysis, we look at a dataset that contains profiles of 3,197 beers from various breweries, with an emphasis on discovering the commonalities between beer genres. Our primary goal is to display the dataset using dimensionality reduction techniques and see if clustering, namely the k-means algorithm, may show meaningful groupings of beer varieties. This approach helps us answer the essential question: can dimensionality reduction and clustering simplify our dataset's complexity enough to expose patterns that are not immediately obvious?



## 2).   **Data loading and preliminary analysis**



The first step in our research is to load the BeerProfiles.csv dataset and perform a preliminary review to determine its structure, variables, and the type of data we're working with. This stage is critical for determining any preprocessing requirements, such as resolving missing values or normalizing data, in order to assure the accuracy of our dimensionality reduction and clustering efforts.

In [7]:
# This cell includes the umap-learn library, which is used for dimension reduction techniques
# that are ideal for visualizing clusters or groups of data points in a low-dimensional space.
!pip install umap-learn

In [8]:
# Imports the necessary Python libraries for data manipulation, scaling, dimensionality reduction, clustering, and visualization
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import plotly.graph_objs as go
import plotly.figure_factory as ff
import umap

In [9]:
# Importing dataset
BeerProfiles_data = pd.read_csv("/content/BeerProfiles.csv")

# Displaying the first few rows of the dataset
print(BeerProfiles_data.head())

# Displaying the shape of the dataset
print(BeerProfiles_data.shape)

# Displaying summary information about the dataset to understand its structure, including data types, non-null counts, and memory usage
print(BeerProfiles_data.info())

# Displaying summary statistics of the dataset
print(BeerProfiles_data.describe())

                           Name    Style  \
0                         Amber  Altbier   
1                    Double Bag  Altbier   
2                Long Trail Ale  Altbier   
3                  Doppelsticke  Altbier   
4  Sleigh'r Dark Doüble Alt Ale  Altbier   

                                            Brewery  ABV  Astringency  Body  \
0                               Alaskan Brewing Co.  5.3           13    32   
1                            Long Trail Brewing Co.  7.2           12    57   
2                            Long Trail Brewing Co.  5.0           14    37   
3  Uerige Obergärige Hausbrauerei GmbH / Zum Uerige  8.5           13    55   
4                           Ninkasi Brewing Company  7.2           25    51   

   Alcohol  Bitter  Sweet  Sour  Salty  Fruits  Hoppy  Spices  Malty  
0        9      47     74    33      0      33     57       8    111  
1       18      33     55    16      0      24     35      12     84  
2        6      42     43    11      0      10 

## 3).  **Data Preprocessing**

After the initial examination, we focus on preprocessing the data. This involves cleaning the data by handling missing values, if any, and standardizing the features to prepare for dimensionality reduction. Standardizing is essential as it ensures that each feature contributes equally to the analysis, preventing variables with larger scales from dominating the results.

In [10]:
# Check the missing values
print(BeerProfiles_data.isnull().sum())

Name           0
Style          0
Brewery        0
ABV            0
Astringency    0
Body           0
Alcohol        0
Bitter         0
Sweet          0
Sour           0
Salty          0
Fruits         0
Hoppy          0
Spices         0
Malty          0
dtype: int64


In [11]:
# Dropping unecessary columns from the dataset because they contain categorical or textual data that are unsuitable for numerical machine learning techniques.
data = BeerProfiles_data.drop(['Name', 'Style', 'Brewery'], axis = 1)

# Display the type and shape of the data after dropping columns
print(type(data))
print(data.shape)

<class 'pandas.core.frame.DataFrame'>
(3197, 12)


In [12]:
# Creates a StandardScaler instance to standardize the dataset's features to zero mean and unit variance
# which is frequently required for optimal performance of many machine learning methods.
feature_scaler= StandardScaler()
scaled_data = feature_scaler.fit_transform(data)

## 4).   **Dimensionality Reduction with PCA**

Our first method for lowering the dimensionality of the dataset is to use Principal Component Analysis (PCA), which enables us to see the data in a two-dimensional space. By analyzing the variance in our dataset, PCA enables us to spot trends among the beer varieties according to their fundamental qualities. To evaluate the amount of information maintained following the reduction, we talk about the variance explained by the PCA components.

In [13]:
# Initialize a PCA object to reduce the dataset dimensions to 2 variables for easy visualization
pca = PCA(n_components = 2)

# Fit PCA on the normalized dataset
pca.fit(scaled_data)

# Transform the data according to the PCA model and print the result
data_pca = pca.transform(scaled_data)

# Prints the variance explained by these components
print("Variance explained by each of the n_components: ",pca.explained_variance_ratio_)
print("Total variance explained by the n_components: ",sum(pca.explained_variance_ratio_))

Variance explained by each of the n_components:  [0.25423708 0.20845063]
Total variance explained by the n_components:  0.462687709379783


In [14]:
# Create scatter plot of the data transformed by PCA, most likely to show how the data points are spread over the two principle components
style=list(BeerProfiles_data['Style'])
data = [go.Scatter(x=data_pca[:,0], y=data_pca[:,1], mode='markers',
                    marker = dict(color='purple', colorscale='Viridis', opacity=0.5),
                                text=[f'Style: {a}' for a in style],
                                hoverinfo='text')]

layout = go.Layout(title = 'PCA Dimensionality Reduction of Beer Styles', width = 800, height = 800,
                    xaxis = dict(title='First Principal Component'),
                    yaxis = dict(title='Second Principal Component'))
fig = go.Figure(data=data, layout=layout)
fig.show()

The dataset's greatest variance is captured by the first principal component (PC1), which also identifies the most important pattern among the variables. The second principal component (PC2), which is perpendicular to the first, reflects the second biggest variance and provides further, unique information about the structure of the data. By highlighting both the primary and secondary patterns in the data, PC1 and PC2 work together to successfully reduce dimensionality and visualize complex datasets, offering a holistic picture.

## 5).  **Exploring UMAP Dimensionality Reduction method**

Uniform Manifold Approximation and Projection (UMAP) is a more adaptable approach to dimensionality reduction that may preserve both local and global data structures better than PCA. Using UMAP, we want to illustrate intricate patterns and interactions between beer varieties, thereby improving our knowledge of the dataset's inherent structure.

In [15]:
# Applies UMAP to reduce dimension to two components, then uses a scatter plot to display the outcome
# Understanding the data's underlying structure and clustering patterns is made easier with the aid of this display
u = umap.UMAP(n_components = 2, n_neighbors=15, min_dist=0.05)
data_umap = u.fit_transform(scaled_data)

style=list(BeerProfiles_data['Style'])
data = [go.Scatter(x=data_umap[:,0], y=data_umap[:,1], mode='markers',
                    marker = dict(color='purple', colorscale='Viridis', opacity=0.5),
                                text=[f'Style: {a}' for a in style],
                                hoverinfo='text')]

layout = go.Layout(title = 'UMAP Dimensionality Reduction of Beer Styles', width = 800, height = 800,
                    xaxis = dict(title='First UMAP Dimension '),
                    yaxis = dict(title='Second UMAP Dimension'))
fig = go.Figure(data=data, layout=layout)
fig.show()

The two-dimensional projection of the Beer dataset is displayed in the "UMAP Dimensionality Reduction of Beer Styles" scatter plot that is the output. Plotting a beer according to its UMAP-transformed attributes results in a point for each beer. The distribution of beers in this condensed area is depicted in the diagram, which may represent both similarities and variations in their profiles. It's possible that beers closer in proximity share more traits than those farther away. Extra context for every data point is provided by the hover tooltips displaying the different beer styles. This data visualization aids in identifying trends and clusters that may guide future research or decision-making concerning the characteristics of different beer varieties.

## 6).   **Clustering with K-Means**

To see if beer styles can be classified into discrete clusters based on their characteristics, we use the k-means clustering technique on our dimensionality-reduced data. This stage entails selecting the best number of clusters and analyzing the results to understand how beer styles are structured within the dataset. Clustering can help us understand the similarities and differences across beer styles, which contributes to the ultimate purpose of our investigation.

In [16]:
# Initiate a KMeans clustering model to divide the data into five groups according to the UMAP findings
kmeans = KMeans(n_clusters = 5)

# Utilizing the UMAP-transformed data, fit the KMeans model
kmeans.fit(data_umap)

# Take the fitted KMeans model and extract the cluster labels
labels = list(kmeans.labels_)

# Create the scatter plot data from UMAP-transformed data, coloring points using KMeans cluster labels
data = [go.Scatter(x=data_umap[:,0], y=data_umap[:,1], mode='markers',
                    marker = dict(color=kmeans.labels_, colorscale='Rainbow', opacity=0.5),
                                text=[f'Style: {a}<br>Label: {b}' for a,b in list(zip(style,labels))],
                                hoverinfo='text')]

# Establish the KMeans clustering scatter plot's arrangement
layout = go.Layout(title = 'UMAP Dimensionality Reduction using K-Mean', width = 800, height = 800,
                    xaxis = dict(title='First Dimension'),
                    yaxis = dict(title='Second Dimension'))

# Create and present a figure depicting UMAP data colored using KMeans clustering
fig = go.Figure(data=data, layout=layout)
fig.show()

# Insert the cluster labels into the original dataset as a new column
BeerProfiles_data['Label'] = kmeans.labels_

# Create a new CSV file with the revised dataset with cluster labels
BeerProfiles_data.to_csv("BeerProfiles.csv", index=False)

# Show the distribution of data points across clusters
print(BeerProfiles_data.Label.value_counts())

/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning



Label
1    892
4    639
2    637
0    591
3    438
Name: count, dtype: int64


The result is a scatter plot, where each hue represents one of the five clusters identified by K-Means and shows how the Beer dataset is clustered in two dimensions. This graphic shows how the beers are arranged according to the underlying patterns in the data that UMAP has identified. For example, one could conclude that beers within a cluster have similar properties, whereas those outside of a cluster have different properties. Understanding the structure of the dataset and using it to inform decisions about product classification, recommendation engines, and targeted advertising in the beer sector can both benefit from this. The distribution of beers among the clusters and the size of each cluster are displayed in the table beneath the plot.

## 7).   **Conclusion and Insights**

dimensionality reduction techniques such as PCA or UMAP can successfully identify the underlying structure of complicated datasets such as BeerProfiles. They can help you see how different beer styles connect to one another. Furthermore, K-Means clustering can improve this research by finding separate groups or clusters of beers, potentially offering insights into the qualities that distinguish different varieties of beer.

 **K-Means Clustering Utility**: K-Means clustering could be very effective after dimensionality reduction because it helps to identify natural groupings within the beers. If beers of a specific style frequently fall into the same cluster, it may indicate that their profiles are similar based on the beer attributes measured.
To confirm the particular approaches employed, analyze the visuals, and make final conclusions, I'd need to look at the Python notebook (Final.ipynb). Unfortunately, I do not currently have the capacity to review the contents of Jupyter notebook files directly. If you can extract the code and findings into a text format, I can help you with a thorough study and conclusion.


## 8).   **References**

i) https://elearning.dbs.ie/pluginfile.php/2110380/mod_resource/content/0/Avg_Temperatures_UMAP_KMeans.py

ii) https://elearning.dbs.ie/pluginfile.php/2109533/mod_resource/content/0/Avg_Temperatures_PCA_UMAP.py

iii) https://elearning.dbs.ie/mod/book/view.php?id=1504924